In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 2 — Data Quality & Exploratory Data Analysis

**Project:** Wholesale Customer Segmentation

Phase 2 - Data Quality & Exploratory Analysis
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 5. Check data quality (confirm zero missing values and zero duplicates)
 6. Perform relevant exploratory analysis (univariate + bivariate EDA)
 7. Select clustering features (6 spend columns; exclude Channel/Region)

Depends on: phase1_setup.py (df_raw, DATA_PATH)
Outputs: PNG figures saved to ./figures/, printed profiling output

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



# Recreate Phase 1 outputs locally so this notebook is standalone.
DATA_PATH = "data/raw/wholesale_customers.csv"
df_raw = pd.read_csv(DATA_PATH)
PROJECT_OBJECTIVE = (
    "Segment wholesale distribution customers into meaningful groups based on "
    "their annual spending across six product categories (Fresh, Milk, Grocery, "
    "Frozen, Detergents_Paper, Delicassen), using unsupervised clustering "
    "(K-Means, cross-checked with Hierarchical Clustering), in order to reveal "
    "actionable customer archetypes for targeted business decisions "
    "(e.g. promotions, delivery/route planning, inventory allocation)."
)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
CATEGORICAL_COLS = ["Channel", "Region"]

sns.set_style("whitegrid")

## SECTION 1: Data Quality Confirmation (Step 5)

In [ ]:
# SECTION 1: Data Quality Confirmation (Step 5)
# ===========================================================================
def confirm_data_quality(df: pd.DataFrame) -> dict:
    """Re-verify missing values and duplicates on the working copy (df_raw
    is never mutated; this section only reads it)."""
    missing = df.isnull().sum()
    duplicates = df.duplicated().sum()
    negative_spend = (df[SPEND_COLS] < 0).sum()
    zero_spend = (df[SPEND_COLS] == 0).sum()
    constant_cols = [c for c in df.columns if df[c].nunique() <= 1]

    print("=" * 70)
    print("SECTION 1: DATA QUALITY CONFIRMATION")
    print("=" * 70)
    print(f"Missing values per column:\n{missing}")
    print(f"\nTotal missing values: {missing.sum()}")
    print(f"Duplicate rows: {duplicates}")
    print(f"Negative spending values by feature: {negative_spend.to_dict()}")
    print(f"Zero spending values by feature: {zero_spend.to_dict()}")
    print(f"Constant columns: {constant_cols}")

    if (missing.sum() == 0 and duplicates == 0 and
        negative_spend.sum() == 0 and zero_spend.sum() == 0 and not constant_cols):
        print("[OK] Core data quality confirmed: no missing values, duplicate rows, invalid negative/zero spending values, or constant columns.")
    else:
        print("[WARNING] Data quality issue detected — review before proceeding.")

    return {
        "missing_total": int(missing.sum()),
        "duplicates": int(duplicates),
        "negative_spend_total": int(negative_spend.sum()),
        "zero_spend_total": int(zero_spend.sum()),
        "constant_columns": constant_cols,
    }

## SECTION 2: Univariate EDA — Histograms (Step 6)

In [ ]:
# SECTION 2: Univariate EDA — Histograms (Step 6)
# ===========================================================================
def plot_histograms(df: pd.DataFrame) -> None:
    """Histogram per spend column to visualize distribution shape/skew."""
    print("\n" + "=" * 70)
    print("SECTION 2: UNIVARIATE EDA — HISTOGRAMS")
    print("=" * 70)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    for i, col in enumerate(SPEND_COLS):
        sns.histplot(df[col], bins=30, kde=True, ax=axes[i], color="steelblue")
        axes[i].set_title(f"Distribution of {col}")
        axes[i].set_xlabel(f"{col} (annual spend)")
        axes[i].set_ylabel("Count")
    fig.suptitle("Univariate EDA: Spend Category Distributions", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/01_histograms_spend_categories.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/01_histograms_spend_categories.png")

## SECTION 3: Univariate EDA — Boxplots (Step 6)

In [ ]:
# SECTION 3: Univariate EDA — Boxplots (Step 6)
# ===========================================================================
def plot_boxplots(df: pd.DataFrame) -> None:
    """Boxplot per spend column to visualize spread and outliers."""
    print("\n" + "=" * 70)
    print("SECTION 3: UNIVARIATE EDA — BOXPLOTS")
    print("=" * 70)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    for i, col in enumerate(SPEND_COLS):
        sns.boxplot(y=df[col], ax=axes[i], color="darkorange")
        axes[i].set_title(f"Boxplot of {col}")
        axes[i].set_ylabel(f"{col} (annual spend)")
    fig.suptitle("Univariate EDA: Spend Category Boxplots (Outlier View)", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/02_boxplots_spend_categories.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/02_boxplots_spend_categories.png")

## SECTION 4: Univariate EDA — Channel & Region Bar Charts (Step 6)

In [ ]:
# SECTION 4: Univariate EDA — Channel & Region Bar Charts (Step 6)
# ===========================================================================
def plot_categorical_bars(df: pd.DataFrame) -> None:
    """Bar charts showing the distribution of Channel and Region."""
    print("\n" + "=" * 70)
    print("SECTION 4: UNIVARIATE EDA — CHANNEL & REGION DISTRIBUTION")
    print("=" * 70)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    channel_counts = df["Channel"].value_counts().sort_index()
    axes[0].bar(channel_counts.index.astype(str), channel_counts.values, color="seagreen")
    axes[0].set_title("Channel Distribution (1 = Horeca, 2 = Retail)")
    axes[0].set_xlabel("Channel")
    axes[0].set_ylabel("Number of Customers")
    for i, v in enumerate(channel_counts.values):
        axes[0].text(i, v + 3, str(v), ha="center")

    region_counts = df["Region"].value_counts().sort_index()
    axes[1].bar(region_counts.index.astype(str), region_counts.values, color="indianred")
    axes[1].set_title("Region Distribution")
    axes[1].set_xlabel("Region")
    axes[1].set_ylabel("Number of Customers")
    for i, v in enumerate(region_counts.values):
        axes[1].text(i, v + 3, str(v), ha="center")

    fig.suptitle("Univariate EDA: Categorical Field Distributions", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/03_channel_region_bars.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/03_channel_region_bars.png")
    print(f"Channel counts: {channel_counts.to_dict()}")
    print(f"Region counts: {region_counts.to_dict()}")

## SECTION 5: Bivariate EDA — Correlation Heatmap (Step 6)

In [ ]:
# SECTION 5: Bivariate EDA — Correlation Heatmap (Step 6)
# ===========================================================================
def plot_correlation_heatmap(df: pd.DataFrame) -> pd.DataFrame:
    """Correlation heatmap across the 6 spend columns to surface
    multicollinearity ahead of Phase 3."""
    print("\n" + "=" * 70)
    print("SECTION 5: BIVARIATE EDA — CORRELATION HEATMAP")
    print("=" * 70)

    corr = df[SPEND_COLS].corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                square=True, ax=ax)
    ax.set_title("Correlation Heatmap: Spend Categories")
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/04_correlation_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/04_correlation_heatmap.png")
    print(f"\nCorrelation matrix:\n{corr.round(2)}")

    strong_pairs = []
    for i, c1 in enumerate(SPEND_COLS):
        for c2 in SPEND_COLS[i + 1:]:
            r = corr.loc[c1, c2]
            if abs(r) >= 0.6:
                strong_pairs.append((c1, c2, round(r, 2)))
    print(f"\nStrong correlations (|r| >= 0.6): {strong_pairs}")
    return corr

## SECTION 6: Bivariate EDA — Pairplot Colored by Channel (Step 6)

In [ ]:
# SECTION 6: Bivariate EDA — Pairplot Colored by Channel (Step 6)
# ===========================================================================
def plot_pairplot_by_channel(df: pd.DataFrame) -> None:
    """Exploratory pairplot of the six spend columns colored by Channel.
    Channel is an exploratory overlay only; it is not used to create clusters."""
    print("\n" + "=" * 70)
    print("SECTION 6: BIVARIATE EDA — PAIRPLOT COLORED BY CHANNEL")
    print("=" * 70)

    plot_df = df[SPEND_COLS + ["Channel"]].copy()
    plot_df["Channel"] = plot_df["Channel"].map({1: "Horeca (1)", 2: "Retail (2)"})

    g = sns.pairplot(plot_df, hue="Channel", palette={"Horeca (1)": "steelblue", "Retail (2)": "darkorange"},
                      diag_kind="hist", plot_kws={"alpha": 0.6, "s": 25})
    g.fig.suptitle("Pairplot of Spend Categories Colored by Channel", y=1.02, fontsize=14)
    g.savefig(f"{FIG_DIR}/05_pairplot_by_channel.png", dpi=150, bbox_inches="tight")
    plt.close(g.fig)
    print(f"[OK] Saved {FIG_DIR}/05_pairplot_by_channel.png")
    print("[NOTE] Channel coloring is exploratory only and is not a clustering input.")

## SECTION 7: Feature Selection for Clustering (Step 7)

In [ ]:
# SECTION 7: Feature Selection for Clustering (Step 7)
# ===========================================================================
def select_clustering_features(df: pd.DataFrame) -> pd.DataFrame:
    """Select the 6 spend columns as clustering features.
    Channel and Region are intentionally excluded (reserved for post-hoc
    cluster profiling in Phase 7), per the assumption stated in Phase 1."""
    print("\n" + "=" * 70)
    print("SECTION 7: FEATURE SELECTION FOR CLUSTERING")
    print("=" * 70)

    features_df = df[SPEND_COLS].copy()

    print(f"Selected clustering features ({len(SPEND_COLS)}): {SPEND_COLS}")
    print(f"Excluded from clustering (reserved for profiling): {CATEGORICAL_COLS}")
    print(f"Resulting feature matrix shape: {features_df.shape}")

    return features_df

## MAIN — run Phase 2 end to end

In [ ]:
# MAIN — run Phase 2 end to end
# ===========================================================================
if __name__ == "__main__":
    quality_report = confirm_data_quality(df_raw)
    plot_histograms(df_raw)
    plot_boxplots(df_raw)
    plot_categorical_bars(df_raw)
    corr_matrix = plot_correlation_heatmap(df_raw)
    plot_pairplot_by_channel(df_raw)
    features_df = select_clustering_features(df_raw)

    print("\n" + "=" * 70)
    print("PHASE 2 COMPLETE")
    print("=" * 70)
    print(f"Data quality: {quality_report}")
    print(f"Clustering feature matrix ready: {features_df.shape} — {list(features_df.columns)}")
    print("[OK] Ready for Phase 3 (Skewness, Outliers & Multicollinearity treatment).")

### Phase 2 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.